<div dir="rtl">

# HM_04 — מעבדה: האם קובץ ההנחיה באמת עוזר? למדוד במקום לנחש

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע · **מפגש 4:** Context Engineering

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

בהרצאה טענו שקובץ הנחיית פרויקט ($x_\text{inst}$) סוגר החלטות שחוזרות בכל בקשה: בלעדיו המודל מנחש, וההסתברות שהתשובה מתאימה למוסכמה $\ell$ היא $g^\text{open}_\ell$; איתו המודל מציית, וההסתברות עולה ל-$g^\text{closed}_\ell$. הערכים בהרצאה היו **להמחשה**. במעבדה הזו תמדדו אותם: תבנו בודק אוטומטי לחמש מוסכמות של פרויקט ה-Todo, ותריצו את אותן חמש בקשות פעם **בלי** קובץ הנחיה ופעם **עם** קובץ שתכתבו.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | ארגז כלים: המודלים מההרצאה | `consistency`, `mixed_g`, `rule_gain` | 7 דק' |
| ב' | בודק מוסכמות | `full_type_hints`, `public_docstrings`, `good_names`, `stdlib_only`, `handles_empty_list` | 10 דק' |
| ג' | הניסוי: בלי קובץ מול עם קובץ | — (הרצות בכלי AI) | 15 דק' |
| ד' | ניתוח | — | 8 דק' |
| ה' | שיפור ומסקנה | — | 5 דק' |
| שיעורי בית | שיחות ארוכות והקשר ממוקד | `conversation_tokens`, `max_requests`, `split_tokens`, `context_metrics`; חישוב ידני | בבית |

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` — **בעצמכם, בלי כלי AI**. (בחלקים ג' ו-ה' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. הבדיקות משתמשות **במספרים מההרצאה** — כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. ההבדל בין התחזית לתוצאה הוא חלק מהציון.
4. **עונים על שאלות הניתוח** במונחי ההרצאה ובסימונים שלה.

## סימונים (כמו בהרצאה)

| סימון | משמעות |
|---|---|
| $x_\text{inst}$ | קובץ הנחיית הפרויקט |
| $\ell,\ g_\ell$ | אינדקס של מוסכמה (החלטה); ההסתברות שתשובה אחת מקיימת אותה |
| $g^\text{open}_\ell,\ g^\text{closed}_\ell$ | $g_\ell$ בלי הכלל (המודל מנחש); $g_\ell$ כשהכלל כתוב בקובץ (המודל מציית) |
| $\hat{g}_\ell = c / R$ | האומד של $g_\ell$: $c$ תשובות קיימו את המוסכמה מתוך $R$ |
| $Q$ | מספר הבקשות בפרויקט |
| $\varepsilon$ | ההסתברות לשכוח לכתוב כלל בפרומפט, כשכותבים אותו ידנית בכל בקשה |
| $q_\ell,\ \Delta_\ell,\ \Delta^\text{stale}_\ell$ | שבר הבקשות שבהן המוסכמה רלוונטית; הרווח הצפוי מכלל (חריגות שנחסכו לבקשה); הרווח מכלל מיושן |
| $L,\ n_\text{inst}$ | אורך חלון ההקשר; מספר הטוקנים של קובץ ההנחיה |
| $K,\ k,\ J$ | מספר הבקשות בשיחה; אינדקס של בקשה בשיחה; מספר השיחות שמפצלים אליהן |
| $n_\text{fixed},\ n_\text{turn},\ n_\text{sum}$ | טוקנים קבועים בכל בקשה; טוקנים שכל חילוף מוסיף; אורך סיכום מעבר |
| $T_k,\ T(K),\ K_\text{max}$ | טוקנים בבקשה $k$; סך הטוקנים בשיחה של $K$ בקשות; הבקשה האחרונה שנכנסת לחלון |
| $\mathcal{N},\ \mathcal{C},\ n_\text{files}$ | הצרכים של המשימה; מה שצורף להקשר; הטוקנים של מה שצורף |

## הגשה

בסוף: **Runtime → Restart session and run all**, ודאו שתא הבדיקה המסכם מראה `✓` לכל הפונקציות ושתאי הניסוי רצים, ואז **File → Download → Download .ipynb** והגישו.

</div>

In [ ]:
# תא הכנה — הריצו פעם אחת בתחילת העבודה
import ast
import math
import re
import sys
import types

# המודולים של פרויקט ה-Todo. תשובה שמייבאת מהם (למשל from dates import parse_date) צריכה לרוץ גם כאן,
# ולכן כל אחד מהם מוחלף במודול ריק: כל שם שמבקשים ממנו הוא פונקציה שלא עושה כלום.
PROJECT_MODULES = {"tasks", "storage", "dates", "main", "models", "export", "config"}


class _StubModule(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        return lambda *args, **kwargs: None


for _name in PROJECT_MODULES:
    sys.modules.setdefault(_name, _StubModule(_name))


# --- מ-HM_03 (נתון): רווח וילסון ומבחן פישר ---
def wilson_interval(c, R, z_star=1.96):
    """Wilson score interval for a success probability, from c successes out of R runs."""
    P_hat = c / R
    z2 = z_star ** 2
    centre = P_hat + z2 / (2 * R)
    half = z_star * math.sqrt(P_hat * (1 - P_hat) / R + z2 / (4 * R ** 2))
    denom = 1 + z2 / R
    return max(0.0, (centre - half) / denom), min(1.0, (centre + half) / denom)


def fisher_p_value(c_A, R_A, c_B, R_B):
    """Two-sided Fisher exact test p-value for c_A/R_A vs c_B/R_B successes."""
    total = c_A + c_B

    def pr(a):
        return math.comb(R_A, a) * math.comb(R_B, total - a) / math.comb(R_A + R_B, total)

    observed = pr(c_A)
    a_min, a_max = max(0, total - R_B), min(R_A, total)
    return sum(pr(a) for a in range(a_min, a_max + 1) if pr(a) <= observed * (1 + 1e-9))


print("✓ מוכן")

<div dir="rtl">

---

# חלק א' — ארגז כלים: המודלים מההרצאה (7 דק')

שלוש נוסחאות מההרצאה (חלקים ב', ד', ה'). בחלק ד' של המעבדה תציבו בהן את מה שמדדתם.

## 1. עקביות לאורך $Q$ בקשות

מוסכמה אחת, $Q$ בקשות בשיחות חדשות (ולכן בלתי-תלויות), בכל אחת הסתברות $g$ לקיים אותה:

$$\Pr[\text{all } Q \text{ answers follow the convention}] = g^{Q}, \qquad \mathbb{E}[\text{violations}] = Q\,(1 - g)$$

**דוגמה ידנית** (הטבלה מההרצאה, $Q = 10$): $g = 0.7$ נותן $0.7^{10} \approx 0.028$ ו-$10 \cdot 0.3 = 3$ חריגות צפויות.

## 2. כותבים את הכלל בפרומפט, ושוכחים

כותבים את הכלל בכל פרומפט ידנית, ושוכחים בהסתברות $\varepsilon$. לפי נוסחת ההסתברות השלמה:

$$g = (1 - \varepsilon)\, g^\text{closed} + \varepsilon\, g^\text{open}$$

**דוגמה ידנית:** $g^\text{open} = 0.7$, $g^\text{closed} = 0.95$, $\varepsilon = 0.2$: $g = 0.8 \cdot 0.95 + 0.2 \cdot 0.7 = 0.9$.

## 3. הרווח הצפוי מכלל — מעודכן ומיושן

כלל רלוונטי לשבר $q$ מהבקשות. **כלל מעודכן** מעלה את ההסתברות מ-$g^\text{open}$ ל-$g^\text{closed}$. **כלל מיושן** (המוסכמה השתנתה והקובץ לא) גורם למודל לכתוב לפי הכלל הישן, ולכן התשובה נכונה רק כשהמודל **לא** מציית:

$$\Delta = q\,\big(g^\text{closed} - g^\text{open}\big), \qquad \Delta^\text{stale} = q\,\big((1 - g^\text{closed}) - g^\text{open}\big)$$

**דוגמה ידנית:** $q = 1$, $g^\text{open} = 0.3$, $g^\text{closed} = 0.95$: $\Delta = 0.65$, ו-$\Delta^\text{stale} = 0.05 - 0.3 = -0.25$ — הכלל המיושן **מוסיף** רבע חריגה בכל בקשה.

</div>

In [ ]:
def consistency(g, Q):
    """One convention over Q independent requests.
    returns: (Pr[all Q answers follow it], E[number of violations]).
    e.g. consistency(0.7, 10) -> approximately (0.028, 3.0)
    """
    # TODO
    raise NotImplementedError



def mixed_g(g_open, g_closed, eps):
    """g when the rule is written by hand in each prompt and forgotten with probability eps."""
    # TODO
    raise NotImplementedError



def rule_gain(q, g_open, g_closed, stale=False):
    """Expected violations saved per request by a rule: q * (g_closed - g_open).
    stale=True: the convention changed but the rule did not -> q * ((1 - g_closed) - g_open).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — המודלים (המספרים מההרצאה)
def check_models():
    def r(t, n=3):
        return tuple(round(v, n) for v in t)
    assert r(consistency(0.7, 10)) == (0.028, 3.0), "consistency: לא כותבים — 0.7^10 ≈ 0.028, 3 חריגות"
    assert r(consistency(0.9, 10)) == (0.349, 1.0), "consistency: בפרומפט ידנית — 0.9^10 ≈ 0.349, חריגה אחת"
    assert r(consistency(0.95, 10)) == (0.599, 0.5), "consistency: קובץ הנחיה — 0.95^10 ≈ 0.599, חצי חריגה"
    assert r(consistency(0.8, 8)) == (0.168, 1.6), "consistency: 'בדקו את עצמכם' — docstrings בלי קובץ, 8 בקשות"
    assert r(consistency(0.97, 8)) == (0.784, 0.24), "consistency: 'בדקו את עצמכם' — docstrings עם קובץ, 8 בקשות"
    assert round(mixed_g(0.7, 0.95, 0.2), 9) == 0.9, "mixed_g: הדוגמה מההרצאה — 0.8·0.95 + 0.2·0.7 = 0.9"
    assert round(mixed_g(0.7, 0.95, 0.0), 9) == 0.95, "mixed_g: בלי שכחה (ε = 0) מקבלים g_closed"
    assert round(mixed_g(0.7, 0.95, 1.0), 9) == 0.7, "mixed_g: שוכחים תמיד (ε = 1) — g_open"
    for q, g_open, g_closed, delta in ((1.0, 0.7, 0.95, 0.25), (0.8, 0.6, 0.95, 0.28), (0.3, 0.7, 0.95, 0.075),
                                       (0.1, 0.6, 0.95, 0.035), (1.0, 0.99, 0.99, 0.0), (0.4, 0.3, 0.95, 0.26)):
        assert round(rule_gain(q, g_open, g_closed), 6) == delta, f"rule_gain: q={q}, g_open={g_open}, g_closed={g_closed} צריך לתת {delta}"
    assert round(rule_gain(1, 0.3, 0.95, stale=True), 6) == -0.25, "rule_gain: כלל מיושן מההרצאה — 0.05 − 0.3 = −0.25"
    assert round(rule_gain(0.5, 0.6, 0.9, stale=True), 6) == -0.25, "rule_gain: 'בדקו את עצמכם' — unittest מיושן"
    assert round(rule_gain(0.5, 0.6, 0.9), 6) == 0.15, "rule_gain: 'בדקו את עצמכם' — הכלל המעודכן (pytest)"
    return True

check_models() and print("✓ consistency, mixed_g, rule_gain")

<div dir="rtl">

---

# חלק ב' — בודק מוסכמות (10 דק')

כדי לאמוד $g_\ell$ צריך לדעת, לכל תשובה, אם היא מקיימת את המוסכמה. בהרצאה (חלק ד') ראינו שכלל טוב הוא **כלל בדיק**. חמש המוסכמות של פרויקט ה-Todo, וכל אחת נבדקת אוטומטית:

| # | מוסכמה | מה נבדק | פונקציה |
|---|---|---|---|
| 1 | type hints מלאים | לכל פונקציה: לכל פרמטר ולערך ההחזרה יש annotation. פרמטר בשם `self` או `cls` פטור | `full_type_hints(tree)` |
| 2 | docstring | לכל פונקציה **ציבורית** (שם שלא מתחיל ב-`_`) יש docstring | `public_docstrings(tree)` |
| 3 | שמות | כל פונקציה ציבורית: snake_case, והמילה הראשונה פועל מהרשימה `VERBS` | `good_names(tree)` |
| 4 | בלי תלויות חיצוניות | כל `import` הוא מהספרייה הסטנדרטית, ממודול של הפרויקט (`PROJECT_MODULES`), או יחסי (`from . import x`) | `stdlib_only(tree)` |
| 5 | רשימה ריקה | הרצת כל פונקציה ציבורית ברמה העליונה על `[]` לא זורקת חריגה (נבדק בהרצה, לא ב-`ast`) | `handles_empty_list(code)` |

הפונקציות 1–4 מקבלות **עץ תחביר** (`tree = ast.parse(code)`). מה שצריך מהמודול `ast`:
- `functions_in(tree)` (נתונה בתא הבא) מחזירה את כל הגדרות הפונקציות בקוד, כולל מתודות ופונקציות פנימיות. לכל אחת: `.name`, `.args`, `.returns` (ה-annotation של ערך ההחזרה, או `None`).
- הפרמטרים: `node.args.posonlyargs + node.args.args + node.args.kwonlyargs`, ועוד `node.args.vararg` ו-`node.args.kwarg` (כל אחד מהם `None` אם אין `*args` או `**kwargs`). לכל פרמטר: `.arg` (השם) ו-`.annotation` (`None` אם חסר).
- `ast.get_docstring(node)` מחזירה `None` כשאין docstring.
- `import a.b as c` הוא `ast.Import` עם `node.names` — רשימה שלכל איבר בה יש `.name` (`"a.b"`). `from a.b import c` הוא `ast.ImportFrom` עם `node.module` (`"a.b"`, או `None` ב-`from . import c`) ו-`node.level` (גדול מ-0 ביבוא יחסי). קובע את החבילה החלק הראשון של השם (`"a"`).
- `sys.stdlib_module_names` היא קבוצת שמות המודולים של הספרייה הסטנדרטית (Python 3.10 ומעלה).
- `re.fullmatch(pattern, s)` — ל-snake_case: אותיות קטנות, ספרות וקו תחתון, מתחיל באות, בלי קו תחתון כפול או בסוף.

**כל פונקציה מחזירה `True` אם הקוד מקיים את המוסכמה.** קוד בלי אף פונקציה רלוונטית מקיים אותה (אין מה להפר); את המקרה של תשובה בלי פונקציות בכלל מטפלת `check_conventions` הנתונה.

**ל-`handles_empty_list(code)`:** מריצים את הקוד במילון ריק (`ns = {}`, `exec(code, ns)`), ואז קוראים לכל פונקציה ציבורית **ברמה העליונה** של הקוד (מ-`ast.parse(code).body`) עם הארגומנט היחיד `[]`. מחזירים `False` אם `exec` נכשל או אם קריאה כלשהי זרקה חריגה. פונקציה שדורשת פרמטר נוסף תיכשל ב-`TypeError` — וזה בכוונה: בבקשות הניסוי כל פונקציה מקבלת רק את רשימת המטלות.

</div>

In [ ]:
# נתון: רשימת הפעלים, פונקציית עזר, וקטעי קוד לבדיקה — בכל אחד (חוץ מ-GOOD) מוסכמה אחת מופרת
VERBS = {"add", "delete", "remove", "edit", "update", "filter", "get", "find", "count", "sort", "group",
         "parse", "format", "load", "save", "mark", "validate", "is", "has", "create", "export", "compute"}


def functions_in(tree):
    """All function definitions in the tree (including methods and nested functions)."""
    return [n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]


GOOD = '''
from collections import Counter
from datetime import date
from dates import parse_date


def get_overdue(tasks: list[dict], today: date | None = None) -> list[dict]:
    """Return the tasks that are not done and whose due date has passed."""
    today = today or date.today()
    return [t for t in tasks if t["due"] and not t["done"] and _due(t) < today]


def count_by_status(tasks: list[dict]) -> dict[str, int]:
    """Count done and open tasks."""
    return dict(Counter("done" if t["done"] else "open" for t in tasks))


def _due(task: dict) -> date:
    return date.fromisoformat(task["due"])
'''

NO_RETURN_HINT = '''
def get_titles(tasks: list[dict]):
    """Return the task titles."""
    return [t["title"] for t in tasks]
'''

NO_PARAM_HINT = '''
def get_titles(tasks) -> list[str]:
    """Return the task titles."""
    return [t["title"] for t in tasks]
'''

STAR_ARGS = '''
def add_tasks(tasks: list[dict], *new) -> list[dict]:
    """Return the list with the new tasks appended."""
    return tasks + list(new)
'''

METHOD = '''
class TaskList:
    def add_task(self, task: dict) -> None:
        self.items.append(task)
'''

NO_DOC = '''
def get_titles(tasks: list[dict]) -> list[str]:
    return [t["title"] for t in tasks]
'''

CAMEL = '''
def getTitles(tasks: list[dict]) -> list[str]:
    """Return the task titles."""
    return [t["title"] for t in tasks]
'''

NOUN_FIRST = '''
def titles_of(tasks: list[dict]) -> list[str]:
    """Return the task titles."""
    return [t["title"] for t in tasks]
'''

EXTERNAL = '''
from dateutil import parser


def parse_due(tasks: list[dict]) -> list:
    """Parse the due dates."""
    return [parser.parse(t["due"]) for t in tasks]
'''

NUMPY = '''
import numpy as np


def count_done(tasks: list[dict]) -> int:
    """Count the tasks that are done."""
    return int(np.sum([t["done"] for t in tasks]))
'''

RELATIVE = '''
import os.path
from . import storage
from .dates import parse_date


def load_tasks(tasks: list[dict]) -> list[dict]:
    """Return the tasks unchanged."""
    return tasks
'''

FIRST = '''
def get_first(tasks: list[dict]) -> dict:
    """Return the first task."""
    return tasks[0]
'''

PERCENT = '''
def get_done_percent(tasks: list[dict]) -> float:
    """Return the percentage of tasks that are done."""
    return 100 * sum(t["done"] for t in tasks) / len(tasks)
'''

EXTRA_PARAM = '''
def filter_by_date(tasks: list[dict], day: str) -> list[dict]:
    """Return the tasks due on the given day."""
    return [t for t in tasks if t["due"] == day]
'''

BROKEN_IMPORT = '''
import not_a_real_module_xyz


def get_titles(tasks: list[dict]) -> list[str]:
    """Return the task titles."""
    return [t["title"] for t in tasks]
'''

print("✓ קטעי הבדיקה נטענו")

In [ ]:
def full_type_hints(tree):
    """True if every parameter (except self / cls) and every return value of every function is annotated."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — type hints
def check_hints():
    p = ast.parse
    assert full_type_hints(p(GOOD)), "full_type_hints: GOOD מקיים את המוסכמה (כולל _due הפרטית)"
    assert not full_type_hints(p(NO_RETURN_HINT)), "full_type_hints: חסר annotation לערך ההחזרה"
    assert not full_type_hints(p(NO_PARAM_HINT)), "full_type_hints: חסר annotation לפרמטר"
    assert not full_type_hints(p(STAR_ARGS)), "full_type_hints: גם *new צריך annotation (node.args.vararg)"
    assert full_type_hints(p(METHOD)), "full_type_hints: self פטור"
    assert full_type_hints(p("x = 1")), "full_type_hints: בלי פונקציות אין מה להפר"
    return True

check_hints() and print("✓ full_type_hints")

In [ ]:
def public_docstrings(tree):
    """True if every public function (name not starting with '_') has a docstring."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — docstring
def check_docs():
    p = ast.parse
    assert public_docstrings(p(GOOD)), "public_docstrings: ל-_due אין docstring, אבל היא פרטית"
    assert not public_docstrings(p(NO_DOC)), "public_docstrings: לפונקציה ציבורית חסר docstring"
    assert not public_docstrings(p(METHOD)), "public_docstrings: גם מתודה ציבורית צריכה docstring"
    assert public_docstrings(p("def _helper(x: int) -> int:\n    return x\n")), "public_docstrings: פונקציה פרטית פטורה"
    return True

check_docs() and print("✓ public_docstrings")

In [ ]:
def good_names(tree, verbs=VERBS):
    """True if every public function name is snake_case and its first word is in verbs."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שמות
def check_names():
    p = ast.parse
    assert good_names(p(GOOD)), "good_names: get_overdue ו-count_by_status תקינים; _due פרטית"
    assert not good_names(p(CAMEL)), "good_names: getTitles אינו snake_case"
    assert not good_names(p(NOUN_FIRST)), "good_names: titles_of לא מתחיל בפועל"
    assert not good_names(p("def Get_titles(t: list) -> list:\n    return t\n")), "good_names: אות גדולה"
    assert not good_names(p("def get__titles(t: list) -> list:\n    return t\n")), "good_names: קו תחתון כפול"
    assert good_names(p("def sort_by_due(t: list) -> list:\n    return t\n")), "good_names: sort_by_due תקין"
    assert good_names(p("def get(t: list) -> list:\n    return t\n")), "good_names: פועל לבד הוא שם תקין"
    return True

check_names() and print("✓ good_names")

In [ ]:
def stdlib_only(tree):
    """True if every import is from the standard library, a project module, or relative."""
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — תלויות
def check_imports():
    p = ast.parse
    assert stdlib_only(p(GOOD)), "stdlib_only: collections ו-datetime מהספרייה הסטנדרטית, dates מהפרויקט"
    assert not stdlib_only(p(EXTERNAL)), "stdlib_only: dateutil אינה בספרייה הסטנדרטית"
    assert not stdlib_only(p(NUMPY)), "stdlib_only: numpy (import ... as ...)"
    assert stdlib_only(p(RELATIVE)), "stdlib_only: os.path תקין (os), ויבוא יחסי (from . / from .dates) הוא מהפרויקט"
    assert not stdlib_only(p("import json, requests\n")), "stdlib_only: requests ברשימה עם json"
    return True

check_imports() and print("✓ stdlib_only")

In [ ]:
def handles_empty_list(code):
    """Runs code, then calls every public top-level function with [] as its only argument.
    True if nothing raised (exec included).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — רשימה ריקה
def check_empty():
    assert handles_empty_list(GOOD), "handles_empty_list: GOOD מחזירה [] ו-{} על רשימה ריקה"
    assert not handles_empty_list(FIRST), "handles_empty_list: tasks[0] על רשימה ריקה — IndexError"
    assert not handles_empty_list(PERCENT), "handles_empty_list: חלוקה ב-len(tasks) — ZeroDivisionError"
    assert not handles_empty_list(EXTRA_PARAM), "handles_empty_list: פונקציה שדורשת פרמטר נוסף נכשלת (TypeError)"
    assert not handles_empty_list(BROKEN_IMPORT), "handles_empty_list: exec שנכשל (ImportError) — False"
    assert handles_empty_list(METHOD), "handles_empty_list: מתודות לא נקראות, רק פונקציות ברמה העליונה"
    return True

check_empty() and print("✓ handles_empty_list")

<div dir="rtl">

## הבודק המשולב (נתון)

`check_conventions(code)` מריצה את חמש הפונקציות שלכם ומחזירה מילון `{מוסכמה: True/False}`. תשובה שאינה Python תקין, או שאין בה אף פונקציה, נכשלת בכל חמש המוסכמות (התשובה לא בפורמט שביקשנו). `record` מריצה את הבודק ושומרת את התוצאה ביומן הניסוי `LOG`.

**תחזית:** על אילו מוסכמות ייכשלו `FIRST`, `EXTERNAL` ו-`NO_DOC`? (הריצו אחרי שכתבתם.)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
CONVENTIONS = ("type_hints", "docstring", "naming", "stdlib_only", "empty_list")


def check_conventions(code):
    """Runs the five checks. An answer that does not parse, or has no function, fails all five."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return dict.fromkeys(CONVENTIONS, False)
    if not functions_in(tree):
        return dict.fromkeys(CONVENTIONS, False)
    return {"type_hints": full_type_hints(tree), "docstring": public_docstrings(tree),
            "naming": good_names(tree), "stdlib_only": stdlib_only(tree), "empty_list": handles_empty_list(code)}


def show(result):
    return "  ".join(("✓ " if ok else "✗ ") + name for name, ok in result.items())


for name in ("GOOD", "FIRST", "EXTERNAL", "NO_DOC"):
    print(f"{name:9}", show(check_conventions(globals()[name])))

<div dir="rtl">

---

# חלק ג' — הניסוי: בלי קובץ מול עם קובץ (15 דק')

## פרוטוקול (קראו לפני שמתחילים)

1. **כל הרצה בשיחה חדשה** — אחרת ההרצות לא בלתי-תלויות: בתוך שיחה המודל ממשיך את הסגנון של התשובות הקודמות (few-shot מתוך ההיסטוריה, ההסתייגות בחלק ב' בהרצאה).
2. **שני מצבים, אותו כלי:**
   - **בלי קובץ (`"open"`):** שולחים רק את הפרומפט.
   - **עם קובץ (`"closed"`):** בכלי שתומך בקובץ הנחיה (Claude Code, Cursor, Copilot) — שמים את הקובץ שלכם בשורש תיקייה ריקה ופותחים שם שיחה חדשה. בצ'אט רגיל — מדביקים את תוכן הקובץ בתחילת ההודעה, ואחריו את הפרומפט. **הפרומפט עצמו זהה בשני המצבים.**
3. **לסירוגין:** בקשה 1 בלי, בקשה 1 עם, בקשה 2 בלי, ... — כך ששינוי בכלי באמצע ישפיע על שני המצבים.
4. **מעתיקים את הקוד כמו שהוא.** ההתאמה היחידה שמותרת: למחוק שורות מחוץ לפונקציות (דוגמאות שימוש, `print`, `input`). הרצה שדרשה התאמה — מסמנים `adapted=True`.
5. **מחליטים מראש:** חמש בקשות לכל מצב ($R = 5$). לא משנים את הקובץ באמצע הניסוי (את זה עושים בחלק ה').

**למה מבנה המטלה כתוב בפרומפט?** כדי שההבדל היחיד בין המצבים יהיה **המוסכמות**. בפרויקט אמיתי גם המידע הזה שייך לקובץ ההנחיה (חלק ג' בהרצאה).

## קובץ ההנחיה שלכם

כתבו את הקובץ **באנגלית** (חלק ג' בהרצאה: פחות טוקנים), לפי ששת המרכיבים. הוא צריך לסגור את חמש המוסכמות **בכללים בדיקים**, כלומר כך שמי שקורא את הכלל יודע בדיוק מה `check_conventions` תבדוק. לפני שכותבים, מלאו את טבלת המיפוי:

| מוסכמה | המשפט בקובץ שסוגר אותה | המרכיב (מתוך השישה) |
|---|---|---|
| 1. type hints | | |
| 2. docstring | | |
| 3. שמות (פועל מהרשימה) | | |
| 4. בלי תלויות חיצוניות | | |
| 5. רשימה ריקה | | |

**תחזית:** לכל מוסכמה — בכמה מתוך 5 התשובות **בלי** הקובץ היא תתקיים ($c^\text{open}$), ובכמה **עם** הקובץ ($c^\text{closed}$)?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
# קובץ ההנחיה שלכם — כתבו אותו כאן (באנגלית). הוא משמש בחלק ג' ובשאלה (ג) בחלק ד'.
INSTRUCTION_FILE = """
# Project: Todo App

(כתבו כאן)
"""

<div dir="rtl">

## הבקשות

חמש הבקשות נתונות. `print_prompt(i)` מדפיסה את הפרומפט המלא של בקשה `i` להעתקה. בכל הבקשות הפונקציה מקבלת **רק** את רשימת המטלות.

## עשר ההרצות

לכל הרצה: שיחה חדשה ← שליחת הפרומפט (במצב `"closed"` — עם הקובץ) ← הדבקת הקוד שהתקבל **בין המירכאות** בתא של ההרצה ← הרצת התא. השורה האחרונה, `record(...)`, בודקת את הקוד ורושמת את התוצאה ב-`LOG`. (אם בקוד שקיבלתם יש `'''`, החליפו אותם ב-`"""`.)

</div>

In [ ]:
REQUESTS = {
    1: "Write a function that sorts the tasks by due date; tasks without a due date go last.",
    2: "Write a function that returns only the tasks that are not done yet.",
    3: "Write a function that returns the task with the earliest due date.",
    4: "Write a function that computes the percentage of tasks that are done.",
    5: "Write a function that groups the tasks by the month of their due date.",
}
TASK_MODEL = ('In a Python Todo app, a task is a dict with the keys "title" (str), '
              '"due" (a "YYYY-MM-DD" string, or None) and "done" (bool). '
              'The function takes only the list of tasks.')


def print_prompt(i):
    print(f"{TASK_MODEL} {REQUESTS[i]} Return only the code.")


LOG = []   # יומן הניסוי: רשומה אחת לכל הרצה


def record(cond, request, code, adapted=False):
    """Checks the answer and appends the result to LOG. cond: "open" (no file) or "closed" (with file)."""
    result = check_conventions(code)
    LOG.append({"cond": cond, "request": request, "result": result, "adapted": adapted})
    note = " (נדרשה התאמת פורמט)" if adapted else ""
    print(f"--- הרצה {len(LOG)}: בקשה {request}, {cond}{note} ---")
    print(show(result))


print_prompt(1)

In [ ]:
# הרצה 1 — בקשה 1, בלי קובץ ("open")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("open", 1, answer, adapted=False)

In [ ]:
# הרצה 2 — בקשה 1, עם קובץ ("closed")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("closed", 1, answer, adapted=False)

In [ ]:
# הרצה 3 — בקשה 2, בלי קובץ ("open")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("open", 2, answer, adapted=False)

In [ ]:
# הרצה 4 — בקשה 2, עם קובץ ("closed")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("closed", 2, answer, adapted=False)

In [ ]:
# הרצה 5 — בקשה 3, בלי קובץ ("open")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("open", 3, answer, adapted=False)

In [ ]:
# הרצה 6 — בקשה 3, עם קובץ ("closed")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("closed", 3, answer, adapted=False)

In [ ]:
# הרצה 7 — בקשה 4, בלי קובץ ("open")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("open", 4, answer, adapted=False)

In [ ]:
# הרצה 8 — בקשה 4, עם קובץ ("closed")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("closed", 4, answer, adapted=False)

In [ ]:
# הרצה 9 — בקשה 5, בלי קובץ ("open")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("open", 5, answer, adapted=False)

In [ ]:
# הרצה 10 — בקשה 5, עם קובץ ("closed")
answer = r'''
# הדביקו כאן את הקוד שהתקבל
'''
record("closed", 5, answer, adapted=False)

<div dir="rtl">

---

# חלק ד' — ניתוח (8 דק')

התא הבא מסכם את `LOG`: לכל מוסכמה $\ell$ — $\hat{g}^\text{open}_\ell$ ו-$\hat{g}^\text{closed}_\ell$ עם רווחי וילסון, $\hat{\Delta}_\ell$ (עם $q_\ell = 1$, בעזרת `rule_gain` שלכם), ערך p של מבחן פישר, ו-$\hat{g}_\ell^{\,Q}$ ל-$Q = 20$ (בעזרת `consistency` שלכם). בשורה האחרונה: בכמה הרצות התקיימו **כל** חמש המוסכמות.

</div>

In [ ]:
def summarize(log, Q=20):
    rows = {cond: [e for e in log if e["cond"] == cond] for cond in ("open", "closed")}
    R = {cond: len(rows[cond]) for cond in rows}
    print(f"R_open = {R['open']}, R_closed = {R['closed']}\n")
    if not (R["open"] and R["closed"]):
        return
    print(f"{'convention':12} | {'g_open [Wilson]':>24} | {'g_closed [Wilson]':>24} | {'Δ (q=1)':>7} | "
          f"{'p_val':>5} | {'g_open^Q':>8} | {'g_closed^Q':>10}")
    for conv in CONVENTIONS:
        c = {cond: sum(e["result"][conv] for e in rows[cond]) for cond in rows}
        g = {cond: c[cond] / R[cond] for cond in rows}
        cell = {cond: f"{c[cond]}/{R[cond]} = {g[cond]:.2f} [{wilson_interval(c[cond], R[cond])[0]:.2f}, "
                      f"{wilson_interval(c[cond], R[cond])[1]:.2f}]" for cond in rows}
        p_val = fisher_p_value(c["open"], R["open"], c["closed"], R["closed"])
        print(f"{conv:12} | {cell['open']:>24} | {cell['closed']:>24} | {rule_gain(1, g['open'], g['closed']):7.2f} | "
              f"{p_val:5.3f} | {consistency(g['open'], Q)[0]:8.2g} | {consistency(g['closed'], Q)[0]:10.2g}")
    all5 = {cond: sum(all(e["result"].values()) for e in rows[cond]) for cond in rows}
    print(f"\nכל חמש המוסכמות: open {all5['open']}/{R['open']}, closed {all5['closed']}/{R['closed']}")
    adapted = {cond: sum(e["adapted"] for e in rows[cond]) for cond in rows}
    print(f"התאמת פורמט נדרשה: open {adapted['open']}, closed {adapted['closed']}")


summarize(LOG)

<div dir="rtl">

## שאלות ניתוח

**(א) איזה כלל שווה את המקום שלו?** איזו מוסכמה הרוויחה הכי הרבה ($\hat{\Delta}$ הגדול ביותר)? לאיזו $\hat{\Delta} \approx 0$ — ומה זה אומר על הכלל שלה, לפי חלק ד' בהרצאה? האם מותר להסיק מ-5 הרצות שהכלל "לא עוזר"? התייחסו לרווחי וילסון ול-$p_\text{val}$.

**תשובה:** _(כתבו כאן)_

**(ב) עקביות לאורך הפרויקט.** לפי $\hat{g}^{\,Q}$ בטבלה: מה הסיכוי שכל 20 הבקשות בפרויקט יקיימו את המוסכמה החלשה ביותר — בלי קובץ ועם קובץ? אם $\hat{g}^\text{closed} = 5/5 = 1$, האם $g^\text{closed} = 1$? הציבו ב-`consistency` את הקצה **התחתון** של רווח וילסון במקום $\hat{g}$, וכתבו מה השתנה.

**תשובה:** _(כתבו כאן)_

**(ג) כמה עולה הקובץ?** אמדו את $n_\text{inst}$ של הקובץ שלכם לפי היחס שנמדד בהרצאה (226 טוקנים ל-157 מילים, בתא שלמטה). כמה טוקנים הוא עולה לאורך $Q = 50$ בקשות, ואיזה אחוז הוא מחלון של $L = 128{,}000$? האם זה שווה את $\sum_\ell \hat{\Delta}_\ell$ שמדדתם?

**תשובה:** _(כתבו כאן)_

**(ד) תקפות.** ציינו לפחות **שני** איומים על תקפות הניסוי (גורמים שאינם הקובץ ועלולים להשפיע על התוצאה), ואיך הייתם מצמצמים כל אחד. אחד מהם חייב להיות **הבודק עצמו**: תנו דוגמה לקוד ש-`check_conventions` שלכם מסמנת לא נכון (מקיים מוסכמה ומסומן ✗, או להפך).

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא חישוב לשאלות (ב) ו-(ג)
def estimate_tokens(text, rho=226 / 157):
    """Rough token count of English text: words * rho (rho measured in the lecture: 226 tokens / 157 words)."""
    return round(len(text.split()) * rho)

<div dir="rtl">

---

# חלק ה' — שיפור ומסקנה (5 דק')

1. **שיפור:** בחרו מוסכמה אחת שנכשלה **עם** הקובץ (אם אין — זו עם $\hat{g}^\text{closed}$ הנמוך ביותר, או זו שהייתם רוצים לבדוק). נסחו את הכלל מחדש כך שיהיה בדיק יותר, או הוסיפו לו דוגמת ניגוד (כמו `get_overdue, not overdue_tasks`). הריצו שוב **פעמיים** את הבקשה שבה הוא נכשל, עם הקובץ המתוקן, ורשמו עם `record("improved", ...)` בתא שלמטה.

   **הכלל לפני ← אחרי, ומה יצא:** _(כתבו כאן)_

2. **מסקנה** בשלושה משפטים: מה ההשערה, מה נמדד, מה מותר להסיק (כולל ההסתייגויות).

   _(כתבו כאן)_

3. **כיול:** השוו את התחזית שלכם מחלק ג' לתוצאה. באיזו מוסכמה טעיתם הכי הרבה, ולמה?

   _(כתבו כאן)_

---

# בונוס

**1. כלל מיושן בפועל.** "שנו את המוסכמה" (נניח שהפרויקט עבר ל-camelCase), אבל השאירו בקובץ את הכלל הישן. הריצו 5 בקשות **עם** הקובץ, ובדקו כמה תשובות מקיימות את המוסכמה **החדשה** (`good_names` עם בדיקה של camelCase). חשבו $\hat{\Delta}^\text{stale}$ עם `rule_gain(..., stale=True)` והשוו למה שמדדתם.

**2. איחוד נתונים:** אחדו את ה-`LOG` שלכם עם של שני חברים (סה"כ $R = 15$ לכל מצב) והריצו שוב את `summarize`. מה קרה לרווחי הסמך ול-$p_\text{val}$? ואיזה איום תקפות **חדש** נוצר כשמאחדים נתונים מכלים שונים?

</div>

In [ ]:
# תא לשיפור (חלק ה') ולבונוס

<div dir="rtl">

---

# שיעורי בית — חלק 1: כמה עולה שיחה ארוכה?

בהרצאה (חלק ז') בנינו מודל מפשט לעלות של שיחה. כאן תממשו אותו ותשתמשו בו.

**ההגדרות (מההרצאה):**

| סימון | הגדרה |
|---|---|
| $n_\text{fixed}$ | הטוקנים שנשלחים בכל בקשה בלי קשר להיסטוריה: הוראות הכלי, קובץ ההנחיה, קבצים קבועים |
| $n_\text{turn}$ | הטוקנים שכל חילוף (הודעה + תשובה) מוסיף להיסטוריה, בממוצע |
| $T_k$ | הטוקנים שהמודל מעבד בבקשה $k$ |
| $n_\text{sum}$ | אורך סיכום המעבר שפותח כל שיחה חוץ מהראשונה; הוא נכנס ל-$n_\text{fixed}$ של השיחה |
| $J$ | מספר השיחות שמפצלים אליהן משימה של $K$ בקשות (שיחות שוות, $J$ מחלק את $K$) |

**הנוסחאות:**

$$T_k = n_\text{fixed} + k \cdot n_\text{turn}, \qquad T(K) = \sum_{k=1}^{K} T_k = K \cdot n_\text{fixed} + n_\text{turn} \cdot \frac{K(K+1)}{2}, \qquad K_\text{max} = \left\lfloor \frac{L - n_\text{fixed}}{n_\text{turn}} \right\rfloor$$

**פיצול:** $J$ שיחות של $K/J$ בקשות. השיחה הראשונה עולה $T(K/J)$; כל אחת מ-$J - 1$ האחרות עולה $T(K/J)$ כש-$n_\text{fixed}$ מוחלף ב-$n_\text{fixed} + n_\text{sum}$.

**דוגמאות ידניות (מההרצאה):**
- $n_\text{fixed} = 2{,}000$, $n_\text{turn} = 1{,}500$, $L = 128{,}000$: $K_\text{max} = \lfloor 126{,}000 / 1{,}500 \rfloor = 84$, ו-$T(40) = 80{,}000 + 1{,}500 \cdot 820 = 1{,}310{,}000$.
- שתי שיחות של 20 עם $n_\text{sum} = 1{,}000$: $355{,}000 + 375{,}000 = 730{,}000$ — כ-56% מהשיחה האחת.

## המשימה

ממשו שלוש פונקציות:
1. `conversation_tokens(K, n_fixed, n_turn)` — מחזירה $T(K)$.
2. `max_requests(L, n_fixed, n_turn)` — מחזירה $K_\text{max}$ (ו-0 אם אפילו הבקשה הראשונה לא נכנסת).
3. `split_tokens(K, J, n_fixed, n_turn, n_sum)` — מחזירה את סך הטוקנים כשמפצלים ל-$J$ שיחות שוות עם סיכום מעבר.

</div>

In [ ]:
def conversation_tokens(K, n_fixed, n_turn):
    """T(K) = K * n_fixed + n_turn * K(K+1)/2."""
    # TODO
    raise NotImplementedError



def max_requests(L, n_fixed, n_turn):
    """K_max = floor((L - n_fixed) / n_turn), and 0 if not even one request fits."""
    # TODO
    raise NotImplementedError



def split_tokens(K, J, n_fixed, n_turn, n_sum):
    """Total tokens for K requests split into J equal conversations (J divides K);
    every conversation after the first starts with a handoff summary of n_sum tokens.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שיעורי בית חלק 1 (המספרים מההרצאה)
def check_hw1():
    assert conversation_tokens(40, 2_000, 1_500) == 1_310_000, "conversation_tokens: T(40) מההרצאה = 1,310,000"
    assert conversation_tokens(20, 2_000, 1_500) == 355_000, "conversation_tokens: T(20) = 355,000"
    assert conversation_tokens(30, 3_000, 2_000) == 1_020_000, "conversation_tokens: 'בדקו את עצמכם' — T(30) = 1,020,000"
    assert conversation_tokens(1, 2_000, 1_500) == 3_500, "conversation_tokens: בקשה אחת — n_fixed + n_turn"
    assert max_requests(128_000, 2_000, 1_500) == 84, "max_requests: מההרצאה — 84"
    assert max_requests(100_000, 3_000, 2_000) == 48, "max_requests: 'בדקו את עצמכם' — 48"
    assert max_requests(3_000, 2_000, 1_500) == 0, "max_requests: אפילו בקשה אחת לא נכנסת — 0"
    assert split_tokens(40, 1, 2_000, 1_500, 1_000) == 1_310_000, "split_tokens: J = 1 — שיחה אחת, בלי סיכום"
    assert split_tokens(40, 2, 2_000, 1_500, 1_000) == 730_000, "split_tokens: שתי שיחות של 20 מההרצאה — 730,000"
    return True

check_hw1() and print("✓ conversation_tokens, max_requests, split_tokens")

<div dir="rtl">

## שאלות

**(א) שלוש שיחות.** חשבו **ביד** את סך הטוקנים כשמפצלים משימה של $K = 60$ בקשות לשלוש שיחות של 20 (עם $n_\text{fixed} = 2{,}000$, $n_\text{turn} = 1{,}500$, $n_\text{sum} = 1{,}000$), והשוו לשיחה אחת של 60. איזה אחוז מהשיחה האחת זה? אמתו עם הפונקציות בתא שלמטה.

**תשובה:** _(כתבו כאן)_

**(ב) הוכחה.** הוכיחו שבפיצול של שיחה של $K$ בקשות ל-$J$ שיחות שוות (בלי סיכום), האיבר הריבועי קטן **בערך** פי $J$. כתבו את היחס המדויק, והראו שהוא שואף ל-$1/J$ כש-$K$ גדל.

**תשובה:** _(כתבו כאן)_

**(ג) עד כמה לפצל?** חשבו בתא שלמטה את `split_tokens(60, J, ...)` לכל מחלק $J$ של 60. איזה $J$ הכי זול לפי המודל? למה זו **לא** המלצה מעשית — מה המודל לא סופר?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא לשאלות (א) ו-(ג)

<div dir="rtl">

---

# שיעורי בית — חלק 2: הקשר ממוקד

בהרצאה (חלק ו') הגדרנו, לכל בחירה $\mathcal{C}$ של מה מצרפים לבקשה, את **הכיסוי**, **הדיוק** ו**הטוקנים**:

$$\text{rec}(\mathcal{C}) = \frac{\#\{\text{needs covered by } \mathcal{C}\}}{|\mathcal{N}|}, \qquad \text{prec}(\mathcal{C}) = \frac{\#\{\text{items in } \mathcal{C} \text{ that cover a need}\}}{|\mathcal{C}|}, \qquad n_\text{files} = \sum_{F \in \mathcal{C}} n_F$$

- **צורך** אומר מה המודל צריך לראות מקובץ: `"full"` (תוכן מלא, כי המשימה משנה את הקובץ) או `"sig"` (חתימות בלבד, כי המשימה רק משתמשת בפונקציה שלו). חתימה היא השורה הראשונה של הפונקציה וה-docstring, בלי הקוד שבתוכה.
- גם לכל קובץ שצירפתם כתוב איך צירפתם אותו: `"full"` (תוכן מלא) או `"sig"` (חתימות בלבד).
- קובץ שצירפתם **מכסה** צורך אם זה אותו קובץ, וצירפתם ממנו לפחות את מה שהצורך דורש: `"full"` מכסה כל צורך, ו-`"sig"` מכסה רק צורך של `"sig"`.
- $n_F$ הוא מספר הטוקנים של קובץ $F$ כפי שצירפתם אותו: `FILES[F][0]` לקובץ מלא, `FILES[F][1]` לחתימות.

`FILES` (נתון, בתא הבא) הוא פרויקט ה-Todo מההרצאה: 12 קבצים, 20,000 טוקנים. הגדלים להמחשה, כמו בהרצאה (10 טוקנים לשורה; חתימות — עשירית מהקובץ).

## המשימה

ממשו `context_metrics(needs, attached, files=FILES)`: `needs` הוא מילון קובץ ← `"full"`/`"sig"` (מה המשימה צריכה), ו-`attached` הוא מילון קובץ ← `"full"`/`"sig"` (מה צירפתם). מחזירה שלישייה `(rec, prec, n_files)`.

</div>

In [ ]:
FILES = {   # file -> (tokens when attached in full, tokens of its signatures only)
    "tasks.py": (1_800, 180), "storage.py": (1_500, 150), "dates.py": (600, 60), "main.py": (2_400, 240),
    "config.py": (600, 60), "cli_help.py": (1_200, 120), "models.py": (1_400, 140), "export.py": (1_500, 150),
    "tests/test_tasks.py": (2_000, 200), "tests/test_storage.py": (2_000, 200), "tests/test_dates.py": (1_500, 150),
    "README.md": (3_500, 350),
}



def context_metrics(needs, attached, files=FILES):
    """needs: file -> "full" / "sig" (what the task needs); attached: file -> "full" / "sig" (what was attached).
    returns: (rec, prec, n_files).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — הטבלה מההרצאה: filter_by_date ב-tasks.py, עם טסט, בעזרת parse_date מ-dates.py
def check_hw2():
    needs = {"tasks.py": "full", "tests/test_tasks.py": "full", "dates.py": "sig"}
    everything = {f: "full" for f in FILES}
    assert sum(v[0] for v in FILES.values()) == 20_000, "FILES: 20,000 טוקנים בסך הכול, כמו בהרצאה"
    assert context_metrics(needs, everything) == (1.0, 0.25, 20_000), "context_metrics: א. הכול, מלא — (1, 0.25, 20,000)"
    rec, prec, n = context_metrics(needs, {"tasks.py": "full"})
    assert (round(rec, 3), prec, n) == (0.333, 1.0, 1_800), "context_metrics: ב. רק tasks.py — (1/3, 1, 1,800)"
    focused = {"tasks.py": "full", "tests/test_tasks.py": "full", "dates.py": "sig"}
    assert context_metrics(needs, focused) == (1.0, 1.0, 3_860), "context_metrics: ג. ממוקד — (1, 1, 3,860)"
    wrong_level = {"tasks.py": "sig", "tests/test_tasks.py": "full", "dates.py": "sig"}
    rec, prec, n = context_metrics(needs, wrong_level)
    assert (round(rec, 3), round(prec, 3), n) == (0.667, 0.667, 2_240), \
        "context_metrics: חתימות לא מכסות צורך של תוכן מלא"
    return True

check_hw2() and print("✓ context_metrics")

<div dir="rtl">

## שאלה

**(ד) חישוב ידני.** המשימה: "הוסף ל-`storage.py` פונקציה שמייצאת את המטלות לקובץ CSV, עם טסט; את התאריכים מעצבים בעזרת `format_date` שכבר קיימת ב-`dates.py`".
1. מה הצרכים $\mathcal{N}$? לכל אחד: האם המודל צריך את התוכן המלא של הקובץ, או רק את החתימות?
2. מלאו ביד טבלה (rec, prec, $n_\text{files}$) לשלוש האפשרויות, ואמתו עם `context_metrics` בתא שלמטה:
   - **א.** `storage.py` ו-`export.py`, מלאים ("export נשמע קשור").
   - **ב.** `storage.py`, `tests/test_storage.py`, `dates.py` ו-`tasks.py`, כולם מלאים ("ליתר ביטחון").
   - **ג.** `storage.py` ו-`tests/test_storage.py` מלאים, והחתימות של `dates.py`.
3. איזו אפשרות הכי טובה, ולמה? מה המודל יעשה באפשרות א', למרות שהיא הזולה מבין השתיים הראשונות?
4. מה $n_\text{files}$ המינימלי שמשיג $\text{rec} = 1$? למה אי אפשר לצרף רק את החתימות של `tests/test_storage.py`?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא לשאלה (ד)

<div dir="rtl">

---

# קריטריוני הצלחה ותוצר להגשה

- ✓ שלוש הפונקציות של חלק א' וחמש הפונקציות של הבודק מומשו **בלי AI**, וכל תאי הבדיקה מדפיסים `✓`.
- ✓ קובץ ההנחיה כתוב באנגלית, וטבלת המיפוי בחלק ג' מראה משפט **בדיק** לכל אחת מחמש המוסכמות.
- ✓ 10 הרצות בשיחות נפרדות, לסירוגין, כולן רשומות ב-`LOG` — כולל `adapted=True` היכן שנדרשה התאמת פורמט.
- ✓ הפרשנות הסטטיסטית נכונה: אין "הוכחנו" כשההבדל לא מובהק, ואין "הכלל לא עוזר" כש-$p_\text{val} > 0.05$; $\hat{g} = 5/5$ אינו $g = 1$.
- ✓ שאלה (א) בחלק ד' מקשרת בין $\hat{\Delta}_\ell$ לבין $q_\ell$ ולפער בין $g^\text{open}_\ell$ ל-$g^\text{closed}_\ell$.
- ✓ לפחות שני איומים על תקפות, אחד מהם על הבודק, עם דרך לצמצם כל אחד.
- ✓ חלק ה': כלל אחד מנוסח מחדש, עם שתי הרצות חוזרות רשומות ב-`LOG`.
- ✓ שיעורי בית: ארבע הפונקציות מומשו בלי AI ותאי הבדיקה שלהן מדפיסים `✓`; לשאלות (א) ו-(ד) יש חישוב ידני ואימות בקוד, לשאלה (ב) — הוכחה, ולשאלה (ג) — טבלה והסבר.

**תוצר להגשה:** המחברת עצמה, אחרי **Restart session and run all** — עם המימושים, קובץ ההנחיה, טבלת המיפוי, עשר ההרצות, פלט `summarize` ותשובות לכל השאלות.

</div>

<div dir="rtl">

---

# בדיקה מסכמת — הריצו לפני ההגשה

</div>

In [ ]:
for name, check in [("consistency, mixed_g, rule_gain", check_models),
                    ("full_type_hints", check_hints),
                    ("public_docstrings", check_docs),
                    ("good_names", check_names),
                    ("stdlib_only", check_imports),
                    ("handles_empty_list", check_empty),
                    ("conversation_tokens, max_requests, split_tokens", check_hw1),
                    ("context_metrics", check_hw2)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")
runs = sum(e["cond"] in ("open", "closed") for e in LOG)
print(f"\nהרצות ביומן הניסוי: {runs} (נדרשות 10)")